# 🛡️ BeltGuard — Konveyer lentasi shikastlarini aniqlash (Training Notebook)

**Colab'da ishga tushirish tartibi:**
1. `Runtime → Change runtime type → T4 GPU` tanlang
2. Kataklarni tartib bilan ishga tushiring
3. O'z videolaringizni `videos/` papkaga yuklang (chap paneldagi Files orqali yoki Google Drive'dan)

Bosqichlar: muhit → video→kadrlar → Roboflow dataset → (o'z annotatsiyangizni qo'shish) → training → baholash → eksport


## 1. Muhitni tayyorlash

In [ ]:
!pip install -q ultralytics roboflow supervision

import ultralytics
ultralytics.checks()  # GPU ko'rinishini tekshiring: Tesla T4 chiqishi kerak


## 2. Google Drive ulash (ixtiyoriy, lekin tavsiya etiladi)

Colab uzilib qolsa checkpoint'lar saqlanib qoladi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/beltguard'
os.makedirs(PROJECT_DIR, exist_ok=True)
print('Loyiha papkasi:', PROJECT_DIR)


## 3. Video → kadrlar (IXTIYORIY — video bo'lmasa tashlab keting)

Bu bosqich faqat o'z stendingizda video yozgan bo'lsangiz kerak. **Video yo'q bo'lsa to'g'ridan-to'g'ri 4-bosqichga o'ting** — tayyor dataset bilan hammasi ishlaydi, demo-video esa 4.1-katakda dataset rasmlaridan yasaladi.

Video bo'lsa: `/content/videos/` ga yuklang, har `FRAME_STEP`-kadr saqlanadi.

> ⚠️ **Test uchun ajratgan videoni bu yerga qo'shmang!** Uni faqat 7-bosqichda baholash uchun ishlatasiz.

In [ ]:
import cv2, glob, os

VIDEO_DIR = '/content/videos'      # videolaringiz shu yerda
FRAMES_DIR = '/content/frames'     # kadrlar shu yerga chiqadi
FRAME_STEP = 7                     # har 7-kadr (30fps da ~4 kadr/sekund)
MAX_BLUR = 60.0                    # xira kadrlarni tashlab ketish chegarasi (Laplacian var)

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)

saved, skipped_blur = 0, 0
for vid_path in sorted(glob.glob(f'{VIDEO_DIR}/*')):
    name = os.path.splitext(os.path.basename(vid_path))[0]
    cap = cv2.VideoCapture(vid_path)
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % FRAME_STEP == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            blur = cv2.Laplacian(gray, cv2.CV_64F).var()
            if blur < MAX_BLUR:
                skipped_blur += 1
            else:
                cv2.imwrite(f'{FRAMES_DIR}/{name}_{i:06d}.jpg', frame,
                            [cv2.IMWRITE_JPEG_QUALITY, 95])
                saved += 1
        i += 1
    cap.release()
    print(f'{name}: qayta ishlandi')

print(f'\nJami saqlandi: {saved} kadr | Xiralik sabab tashlandi: {skipped_blur}')
print('Endi bu kadrlarni Roboflow ga yuklab, SAM bilan annotatsiya qiling:')
print('https://app.roboflow.com -> Create Project (Instance Segmentation) -> Upload -> Smart Polygon (SAM)')


## 4. Tayyor ochiq datasetni yuklash (Roboflow)

`conveyor-belt-damage` — 922 rasm, 9 klass, CC BY 4.0.
API kalitni olish: app.roboflow.com → Settings → API Key (bepul akkaunt yetadi).

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'SIZNING_API_KALITINGIZ'   # <-- almashtiring!

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('sample-wy2mp').project('conveyor-belt-damage')
dataset = project.version(1).download('yolov8', location='/content/ds_public')
print('Yuklandi:', dataset.location)


In [ ]:
# data.yaml ni ko'rib chiqamiz — klasslar ro'yxati
!cat /content/ds_public/data.yaml


### 4.1. Dataset rasmlaridan demo-video yasash (o'z videongiz bo'lmasa)

Test to'plamidagi rasmlardan `/content/test_video.mp4` yig'iladi — 7-bosqichda baholashda ishlatiladi, oxirida esa yuklab olib lokal dashboard'da "Manba" sifatida ko'rsatasiz.

In [ ]:
import cv2, glob, os

# test yo'q bo'lsa valid dan olamiz
img_dir = next(d for d in ('/content/ds_public/test/images',
                           '/content/ds_public/valid/images') if os.path.isdir(d))
imgs = sorted(glob.glob(f'{img_dir}/*.jpg'))
print(f'{len(imgs)} ta rasm topildi: {img_dir}')

W, H, FPS, HOLD = 640, 640, 10, 15   # har rasm 1.5 soniya (15 kadr / 10 fps)
out = cv2.VideoWriter('/content/test_video.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))
for p in imgs:
    frame = cv2.resize(cv2.imread(p), (W, H))
    for _ in range(HOLD):
        out.write(frame)
out.release()
print('Tayyor: /content/test_video.mp4')

## 5. O'z annotatsiyangizni qo'shish (train v2 uchun)

Roboflow'da o'z kadrlaringizni annotatsiya qilib bo'lgach, **Export → YOLOv8** formatida yuklab oling
va quyidagi katakdagi workspace/project nomlarini o'zingiznikiga almashtiring.

Ikkala datasetni birlashtirish eng oson yo'li — **Roboflow'ning o'zida**: o'z loyihangizga
public datasetni ham import qiling (Universe sahifasida "Clone/Fork" tugmasi bor), klasslarni
moslashtiring (masalan `Tear` → `tear`) va bitta versiya qilib eksport qiling.
Shunda quyida faqat bitta dataset yuklaysiz.

In [ ]:
# O'z birlashtirilgan datasetingiz (annotatsiya tugagach):
# my_project = rf.workspace('SIZNING_WORKSPACE').project('SIZNING_PROJECT')
# my_dataset = my_project.version(1).download('yolov8', location='/content/ds_own')
# DATA_YAML = '/content/ds_own/data.yaml'

# Hozircha faqat public dataset bilan boshlaymiz:
DATA_YAML = '/content/ds_public/data.yaml'
print('Training uchun:', DATA_YAML)


## 6. Training

**v1 (baseline):** avval 60 epoch, tez natija olish uchun. T4 da ~40–60 daqiqa.
Natija yaxshi bo'lsa, tunda `epochs=150, imgsz=960` bilan v2 ni qo'ying.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s-seg.pt')   # COCO pretrained segmentatsiya modeli

results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,            # v2 da 960 qiling — ingichka yirtiqlar uchun yaxshiroq
    batch=16,             # OOM xato chiqsa 8 ga tushiring
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    # Lenta uchun sozlangan augmentatsiya:
    mosaic=0.5,
    copy_paste=0.3,       # segmentatsiyada juda samarali
    degrees=5,            # lenta gorizontal - kuchli burish noreal
    flipud=0.0,           # vertikal flip noreal
    fliplr=0.5,
    hsv_v=0.5,            # yorug'lik o'zgarishiga chidamlilik - MUHIM
    hsv_s=0.4,
    translate=0.2,
    scale=0.5,
    erasing=0.3,
    project=f'{PROJECT_DIR}/runs',   # Drive ga saqlanadi - Colab uzilsa ham qolgan bo'ladi
    name='v1_baseline',
)


## 7. Baholash — metrikalar va vizual tekshiruv

In [ ]:
BEST = f'{PROJECT_DIR}/runs/v1_baseline/weights/best.pt'
model = YOLO(BEST)

# Validation to'plamida metrikalar
metrics = model.val(data=DATA_YAML)
print(f'Segment mAP@50    : {metrics.seg.map50:.3f}')
print(f'Segment mAP@50-95 : {metrics.seg.map:.3f}')
print('\nHar klass bo`yicha mAP@50-95 (segment):')
for idx, cls_name in metrics.names.items():
    if idx < len(metrics.seg.maps):
        print(f'  {cls_name:20s}: {metrics.seg.maps[idx]:.3f}')


In [ ]:
# Confusion matrix va PR-curve rasmlari training papkasida:
import glob as g
from IPython.display import Image, display
for p in g.glob(f'{PROJECT_DIR}/runs/v1_baseline/*.png')[:6]:
    print(p)
    display(Image(p, width=700))


In [ ]:
# PT fayli dashboard uchun yetarli (Ultralytics to'g'ridan-to'g'ri o'qiydi).
print('Yuklab olinadi:', BEST)

from google.colab import files
files.download(BEST)                      # -> lokalda models\best.pt ga qo'ying

# Demo-videoni ham yuklab olamiz (dashboard'da "Manba" sifatida ishlatasiz):
if os.path.exists('/content/test_video.mp4'):
    files.download('/content/test_video.mp4')

## 8. Eksport — dashboard uchun

`best.pt` faylini yuklab oling va dashboard yonidagi `models/` papkaga qo'ying.

In [ ]:
# PT fayli dashboard uchun yetarli (Ultralytics to'g'ridan-to'g'ri o'qiydi).
# Tezlik kerak bo'lsa ONNX ham eksport qilamiz:
model.export(format='onnx', opset=12, imgsz=640)
print('Tayyor! Yuklab olish uchun:')
print(BEST)

from google.colab import files
files.download(BEST)   # brauzer orqali yuklab olish


## 9. Keyingi qadamlar

- ✅ `best.pt` → dashboard `models/` papkasiga
- 🌙 Tunda: `epochs=150, imgsz=960, name='v2_night'` bilan qayta training
- 📊 Slayd uchun: yuqoridagi mAP jadvali + confusion matrix + PR curve rasmlari
- 🎥 Zaxira: test video natijasini (annotatsiyalangan) saqlab qo'ying — demo ishlamasa ko'rsatasiz
